Name: Rohit Ramamoorthy

Student ID: 24246438

DW_Project_2_Data_Cleaning

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Project2_dataset.csv')

In [3]:
plane_types = (
    df['Plane Name']
    .dropna()
    .str.split(';')
    .explode()
    .str.strip()
)

distinct_plane_types = plane_types.nunique()

distinct_plane_types

114

In [4]:
sorted(plane_types.unique())

['Aerospatiale/Alenia ATR 42-300',
 'Aerospatiale/Alenia ATR 42-500',
 'Aerospatiale/Alenia ATR 42-600',
 'Aerospatiale/Alenia ATR 72',
 'Airbus A300-600',
 'Airbus A310',
 'Airbus A318',
 'Airbus A319',
 'Airbus A320',
 'Airbus A321',
 'Airbus A330',
 'Airbus A330-200',
 'Airbus A330-300',
 'Airbus A340',
 'Airbus A340-200',
 'Airbus A340-300',
 'Airbus A340-500',
 'Airbus A340-600',
 'Airbus A380',
 'Airbus A380-800',
 'Antonov AN-140',
 'Antonov AN-24',
 'Antonov An-148',
 'Antonov An-158',
 'Avro RJ100',
 'Avro RJ85',
 'BAe 146',
 'BAe 146-100',
 'BAe 146-200',
 'BAe 146-300',
 'Beechcraft 1900',
 'Bell 212',
 'Boeing 717',
 'Boeing 737',
 'Boeing 737-200',
 'Boeing 737-300',
 'Boeing 737-400',
 'Boeing 737-500',
 'Boeing 737-600',
 'Boeing 737-700',
 'Boeing 737-800',
 'Boeing 737-900',
 'Boeing 747',
 'Boeing 747-400',
 'Boeing 747SP',
 'Boeing 757',
 'Boeing 757-200',
 'Boeing 757-300',
 'Boeing 767',
 'Boeing 767-200',
 'Boeing 767-300',
 'Boeing 767-400',
 'Boeing 777',
 'Boei

In [5]:
# Departure airport attributes.
departure_airports = df[['Departure Airport Name',
                          'Departure Airport City',
                          'Departure Airport Country/Region']].copy()
departure_airports.columns = ['airport_name', 'airport_city',
                              'airport_country_region']

# Arrival airport attributes.
arrival_airports = df[['Arrival Airport Name',
                        'Arrival Airport City',
                        'Arrival Airport Country/Region']].copy()
arrival_airports.columns = ['airport_name', 'airport_city',
                            'airport_country_region']

#Combine departure and arrival airport records to identify airport and country mismatches.
airport = pd.concat([departure_airports, arrival_airports], ignore_index=True)

#Identify airport name and city combinations linked to more than one country/region.
airport_mismatches = (
    airport.groupby(["airport_name", "airport_city"])["airport_country_region"]
    .agg(
        countries_found=lambda x: sorted(x.dropna().unique()),
        country_count=lambda x: x.dropna().nunique(),
        total_records="count"
    )
    .reset_index()
)

#Keep only airport name and city combinations that have more than one country/region.
airport_mismatches = airport_mismatches[airport_mismatches["country_count"] > 1]

airport_mismatches

,airport_name,airport_city,countries_found,country_count,total_records
62,Albany Airport,Albany,"[Australia, United States]",2,6
66,Alberto Carnevalli Airport,Merida,"[Mexico, Venezuela]",2,4
73,Alexandria International Airport,Alexandria,"[Egypt, United States]",2,8
140,Arturo Michelena International Airport,Valencia,"[Spain, Venezuela]",2,14
159,Atlas Brasil Cantanhede Airport,Boa Vista,"[Brazil, Cape Verde]",2,6
286,Birmingham-Shuttlesworth International Airport,Birmingham,"[United Kingdom, United States]",2,30
445,Charles M. Schulz Sonoma County Airport,Santa Rosa,"[Argentina, United States]",2,12
454,Cheddi Jagan International Airport,Georgetown,"[Cayman Islands, Guyana]",2,16
498,Cibao International Airport,Santiago,"[Dominican Republic, Spain]",2,20
515,Cochin International Airport,Kochi,"[India, Japan]",2,100


The data cleaning would emphasise on only on the arrival airport country/region attribute as the departure airport country/region attribute seemed already cleaned.

Since the detected airport name to country name mismatches were found in the arrival airport country field, the correction logic was applied to the arrival airport country/region attribute. The correct values remain unchanged, while incorrect values are corrected by identifying them individually.

The arrival airport country/region had a lot of mismatched countries with the city and airport names and they were corrected while referencing the Airline Dataset Updated -v2.csv from Project - 1.

In [6]:
#Correction mapping for airport name and city combinations with the correct airport country/region.
corrections = {
    ('Albany Airport', 'Albany'): 'Australia',
    ('Alberto Carnevalli Airport', 'Merida'): 'Venezuela',
    ('Alexandria International Airport', 'Alexandria'): 'United States',
    ('Arturo Michelena International Airport', 'Valencia'): 'Venezuela',
    ('Atlas Brasil Cantanhede Airport', 'Boa Vista'): 'Brazil',
    ('Birmingham-Shuttlesworth International Airport', 'Birmingham'): 'United States',
    ('Charles M. Schulz Sonoma County Airport', 'Santa Rosa'): 'United States',
    ('Cheddi Jagan International Airport', 'Georgetown'): 'Guyana',
    ('Cibao International Airport', 'Santiago'): 'Dominican Republic',
    ('Cochin International Airport', 'Kochi'): 'India',
    ('Comodoro Arturo Merino Benitez International Airport', 'Santiago'): 'Chile',
    ('El Alto International Airport', 'La Paz'): 'Bolivia',
    ('Eugene F. Correira International Airport', 'Georgetown'): 'Guyana',
    ('F. D. Roosevelt Airport', 'Oranjestad'): 'Netherlands Antilles',
    ('Florence Regional Airport', 'Florence'): 'United States',
    ('Fort Smith Regional Airport', 'Fort Smith'): 'United States',
    ('Futuna Airport', 'Futuna Island'): 'Vanuatu',
    ('General Jose Antonio Anzoategui International Airport', 'Barcelona'): 'Venezuela',
    ('JAGS McCartney International Airport', 'Cockburn Town'): 'Turks and Caicos Islands',
    ('London City Airport', 'London'): 'United Kingdom',
    ('London Gatwick Airport', 'London'): 'United Kingdom',
    ('London Heathrow Airport', 'London'): 'United Kingdom',
    ('London Luton Airport', 'London'): 'United Kingdom',
    ('London Stansted Airport', 'London'): 'United Kingdom',
    ('Mayor Buenaventura Vivas International Airport', 'Santo Domingo'): 'Venezuela',
    ('Norman Manley International Airport', 'Kingston'): 'Jamaica',
    ('Norman Y. Mineta San Jose International Airport', 'San Jose'): 'United States',
    ('Northwest Florida Beaches International Airport', 'Panama City'): 'United States',
    ('Presidente Joao Batista Figueiredo Airport', 'Sinop'): 'Brazil',
    ('Richmond Airport', 'Richmond'): 'Australia',
    ('San Jose Airport', 'San Jose'): 'Philippines',
    ('Santa Fe Municipal Airport', 'Santa Fe'): 'United States',
    ('Santa Rosa International Airport', 'Santa Rosa'): 'Ecuador',
    ('St Petersburg Clearwater International Airport', 'St. Petersburg'): 'United States',
    ('St Pierre Airport', 'St. -pierre'): 'Saint Pierre and Miquelon',
    ('Sydney Kingsford Smith International Airport', 'Sydney'): 'Australia',
    ('Tri-Cities Regional TN/VA Airport', 'Bristol'): 'United States',
    ('Victoria Regional Airport', 'Victoria'): 'United States',
    ('Waterloo Regional Airport', 'Waterloo'): 'Canada',
}

Airport such as Comodoro Arturo Merino Benitez International Airport, General Jose Antonio Anzoategui International Airport, and Presidente Joao Batista Figueiredo Airport had accent in the data, and it was manually verified with while refencing the Airline Dataset Updated -v2.csv.

There were a few ambiguous arrival airport country/region such as for the airport Santa Ana Airport where in the Project2_dataset.csv the country was the United States. Whereas in the Airline Dataset Updated -v2.csv the arrival airport country/region was mentioned either Solomon Islands or Colombia.  

Hence, these types of ambiguous cases were not updated. And there are similar cases like these that were encountered and not updated as there could genuinely be two countries that have the same airport name.

In [7]:
#Function to correct mismatched arrival airport country/region values.
def country_mismatch(row):
    #Check whether the arrival airport name and city exist in the correction mapping.
    key = (row['Arrival Airport Name'], row['Arrival Airport City'])
    #Keep the original data, if not part the of issues.
    return corrections.get(key, row['Arrival Airport Country/Region'])

#Apply the corrections to the Arrival Airport Country/Region column.
df['Arrival Airport Country/Region'] = df.apply(country_mismatch, axis=1)

## Save the cleaned dataset.
df.to_csv('Project2_dataset_updated.csv', index=False)
print("'Project2_dataset_updated.csv' has been created.")

'Project2_dataset_updated.csv' has been created.


A function to correct mismatched arrival airport country/region values. It checks whether the arrival airport name and city exist in the correction mapping. It keeps the original data, if not part of the issue. If not, it applies the corrections to the Arrival Airport Country/Region column. The cleaned data is then saved in a new csv.

Creating the node for Airline.

In [8]:
#Create the Airline node CSV by keeping distinct airline name and country combinations.
airline = df[['Airline Name', 'Airline Country']].drop_duplicates()

airline.columns = [
    'airline_name',
    'airline_country'
    ]

airline.to_csv('airline.csv', index=False)
print("'airline.csv' has been created.")

'airline.csv' has been created.


Creating the node for Airport.

In [9]:
#Recreate airport records after cleaning the arrival airport country/region values.
departure_airports = df[['Departure Airport Name',
                         'Departure Airport City',
                         'Departure Airport Country/Region']].copy()

departure_airports.columns = [
    'airport_name',
    'airport_city',
    'airport_country_region'
]

arrival_airports = df[['Arrival Airport Name',
                       'Arrival Airport City',
                       'Arrival Airport Country/Region']].copy()

arrival_airports.columns = [
    'airport_name',
    'airport_city',
    'airport_country_region'
]

airport = pd.concat([departure_airports, arrival_airports], ignore_index=True)

#Create the Airport node CSV by combining departure and arrival airports and removing exact duplicates.
airport = airport.drop_duplicates(subset=['airport_name', 'airport_city', 'airport_country_region']).reset_index(drop=True)
airport.to_csv('airport.csv', index=False)
print("'airport.csv' has been created.")

'airport.csv' has been created.


The SERVES_FROM relationship connects each Airline node to the departure Airport nodes it serves from. This relationship is created using the airline name and the departure airport details such as departure airport name, city and country/region.

In [10]:
#Create the SERVES_FROM relationship CSV between Airline and departure Airport..
serves_from = df[[    'Airline Name',
    'Departure Airport Name',
    'Departure Airport City',
    'Departure Airport Country/Region'
]].drop_duplicates()

serves_from.columns = [
    'airline_name',
    'airport_name',
    'airport_city',
    'airport_country_region'
]

serves_from.to_csv('serves_from.csv', index=False)
print("serves_from.csv has been created.")

serves_from.csv has been created.


The ROUTE relationship connects a departure Airport node to an arrival Airport node. Both endpoints use the same Airport label, and the relationship direction represents travel from departure airport to arrival airport. Route-specific attributes such as plane_type and airline_name are stored on the ROUTE relationship.

In [11]:
# ROUTE relationship exists between two airport nodes.
route = df[[
    'Departure Airport Name',
    'Departure Airport City',
    'Departure Airport Country/Region',
    'Arrival Airport Name',
    'Arrival Airport City',
    'Arrival Airport Country/Region',
    'Plane Name',
    'Airline Name'
]].copy()

route.columns = [
    'departure_airport_name',
    'departure_airport_city',
    'departure_airport_country_region',
    'arrival_airport_name',
    'arrival_airport_city',
    'arrival_airport_country_region',
    'plane_type',
    'airline_name'
]

route.to_csv('route.csv', index=False)
print("route.csv has been created.")

route.csv has been created.


In [12]:
from google.colab import files

files.download("airline.csv")
files.download("airport.csv")
files.download("serves_from.csv")
files.download("route.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>